**About the environment:** this notebook is written for **SageMath** and uses the module `hilbert_kunz.py`, which must be in the same folder.

# The Han–Monsky algorithm for diagonal hypersurfaces

This notebook accompanies Sections 1, 3 and 4 of the notes [Notes]. It computes the **Hilbert–Kunz function**, the **Hilbert–Kunz multiplicity**, the **F-signature function** and the **F-signature** of a diagonal hypersurface
$$R=\mathbb F_p[[x_1,\dots,x_n]]\big/\big(x_1^{d_1}+\cdots+x_n^{d_n}\big),\qquad \dim R=n-1,$$
with the method of Han and Monsky [HM93]. By [HM93] ([Notes, Theorem 1.7]) there are $e_0\ge0$, $\mu\ge1$ and $\ell\ge0$ with
$$\operatorname{HK}_e(R)=c\,p^{(n-1)e}+c_1(e)\,\ell^{\lfloor e/\mu\rfloor}\qquad(e\ge e_0),$$
where $c=e_{\mathrm{HK}}(R)$ is rational and $c_1$ is periodic of period $\mu$. The Han–Monsky ring $\Gamma_k$ is the subject of the notebook `01_representation_ring.ipynb`.

The main function is `HK_function_diagonal(f)`, which takes the polynomial itself. Run the next cell, then any of the examples. How the algorithm works is explained afterwards (§3), following [Notes, §3].

In [1]:
load("hilbert_kunz.py")

## 1. Examples

A diagonal hypersurface $x_1^{d_1}+\dots+x_n^{d_n}$ is given by its exponent list `exps = [d_1, ..., d_n]`.

### 1.1 Han–Monsky's example [HM93], [Notes, Example 1.3]
$e_{\mathrm{HK}}$ and the Hilbert–Kunz function of $\mathbb F_5[[x,y,z,t]]/(x^4+y^4+z^4+t^4)$:
$$\operatorname{HK}_e(R)=\tfrac{168}{61}\,5^{3e}-\tfrac{107}{61}\,3^e\qquad(e\ge0).$$

In [2]:
S.<x,y,z,t> = GF(5)[]
HK_function_diagonal(x^4 + y^4 + z^4 + t^4)

HK_e = 168/61 * 5^(3e) - 107/61 * 3^e   for e >= 0

### 1.2 Monsky's periodic example [Mon83], [Notes, Example 1.1]
For $R=\mathbb F_2[[x,y]]/(x^5+y^5)$, $\operatorname{HK}_e(R)=5\cdot2^e-4$ for even $e$ and $5\cdot2^e-6$ for odd $e$. Here $\ell=1$ and $\mu=2$, so the correction term is periodic. The result `H` can also be evaluated; compare with a direct Gröbner-basis computation:

In [3]:
S.<x,y> = GF(2)[]
H = HK_function_diagonal(x^5 + y^5)
H

HK_e = 5 * 2^e - 4   for even e >= 0
HK_e = 5 * 2^e - 6   for odd e >= 1

In [4]:
[H(e) for e in range(7)], [HK_groebner(x^5 + y^5, e) for e in range(7)]

([1, 4, 16, 34, 76, 154, 316], [1, 4, 16, 34, 76, 154, 316])

### 1.3 The size of $\ell$
[Notes, Theorem 1.7] also says: if $n\ge3$ then $0\le\ell\le p^{(n-3)\mu}$, and if $p=2$ or $n\in\{2,3\}$ then $\ell\in\{0,1\}$. Here $\ell$ is the number $\ell^\sharp$ of §3 (`lsh` in the code). A few cases:

In [5]:
for p, exps in [(2,[5,5]), (7,[3,3,3]), (5,[4,4,4,4]), (7,[3,3,3,3]), (3,[6,6,6,5]), (2,[3,3,3,3]),
                (5,[4,4,4,4,4]), (7,[2,2,2,2,2,2])]:
    D = eHK_diagonal(p, exps, _data=True)
    n, mu, lsh = len(exps), D['mu'], D['lsh']
    bound = p^((n-3)*mu) if n >= 3 else 1
    print("p = %d, exps = %s:  mu = %d, e0 = %d, l# = %d,  bound p^((n-3)mu) = %d" % (p, exps, mu, D['e0'], lsh, bound))

p = 2, exps = [5, 5]:  mu = 2, e0 = 0, l# = 1,  bound p^((n-3)mu) = 1
p = 7, exps = [3, 3, 3]:  mu = 1, e0 = 0, l# = 1,  bound p^((n-3)mu) = 1
p = 5, exps = [4, 4, 4, 4]:  mu = 1, e0 = 0, l# = 3,  bound p^((n-3)mu) = 5
p = 7, exps = [3, 3, 3, 3]:  mu = 1, e0 = 0, l# = 5,  bound p^((n-3)mu) = 7
p = 3, exps = [6, 6, 6, 5]:  mu = 2, e0 = 1, l# = 3,  bound p^((n-3)mu) = 9
p = 2, exps = [3, 3, 3, 3]:  mu = 1, e0 = 0, l# = 1,  bound p^((n-3)mu) = 2
p = 5, exps = [4, 4, 4, 4, 4]:  mu = 2, e0 = 0, l# = 18,  bound p^((n-3)mu) = 625
p = 7, exps = [2, 2, 2, 2, 2, 2]:  mu = 1, e0 = 0, l# = 119,  bound p^((n-3)mu) = 343


### 1.4 A period-2 example [Yos09]

In [6]:
S.<x,y,z,t> = GF(3)[]
H = HK_function_diagonal(x^6 + y^6 + z^6 + t^5)
H

HK_e = 459/121 * 3^(3e) - 1548/121 * 3^(e/2)         for even e >= 2
HK_e = 459/121 * 3^(3e) - 2592/121 * 3^((e - 1)/2)   for odd e >= 1

In [7]:
# the formula at e = 4 against a direct computation
H(4), HK_groebner(x^6 + y^6 + z^6 + t^5, 4)

(2015847, 2015847)

### 1.5 F-signature: $x^3+y^3+z^3$ [Notes, Example 4.3]
Let $J=(x,y,z^2)$. By [Notes, Lemma 4.2], $\operatorname{FS}_e(R)=3p^{2e}-\operatorname{HK}_e(R,J)$. For $p\equiv1\pmod 3$ the notes show $\operatorname{HK}_e(R,J)=3p^{2e}-1$, so $\operatorname{FS}_e(R)=1$ ($R$ is F-split but not strongly F-regular). For $p\equiv2\pmod3$, $R$ is not F-pure and $\operatorname{FS}_e(R)=0$ for $e\ge1$. The last column is an independent Gröbner-basis computation of $\operatorname{FS}_e$ (`fs_brute`, notebook 03). The F-signature function itself is printed by `FS_function_diagonal`, in the cell after.

In [8]:
for p in [7, 13, 19, 5, 11]:
    HKJ = [_hk_value(p, [3,3,3], [1,1,2], e) for e in range(3)]
    print("p = %2d (p mod 3 = %d):  HK_e(R,J) = %s,  3p^(2e)-1 = %s,  FS_e = %s,  Groebner FS_e = %s"
          % (p, p % 3, HKJ, [3*p^(2*e) - 1 for e in range(3)],
             [3*p^(2*e) - HKJ[e] for e in range(3)], [fs_brute(p, [[3],[3],[3]], e) for e in range(3)]))

p =  7 (p mod 3 = 1):  HK_e(R,J) = [2, 146, 7202],  3p^(2e)-1 = [2, 146, 7202],  FS_e = [1, 1, 1],  Groebner FS_e = [1, 1, 1]
p = 13 (p mod 3 = 1):  HK_e(R,J) = [2, 506, 85682],  3p^(2e)-1 = [2, 506, 85682],  FS_e = [1, 1, 1],  Groebner FS_e = [1, 1, 1]
p = 19 (p mod 3 = 1):  HK_e(R,J) = [2, 1082, 390962],  3p^(2e)-1 = [2, 1082, 390962],  FS_e = [1, 1, 1],  Groebner FS_e = [1, 1, 1]
p =  5 (p mod 3 = 2):  HK_e(R,J) = [2, 75, 1875],  3p^(2e)-1 = [2, 74, 1874],  FS_e = [1, 0, 0],  Groebner FS_e = [1, 0, 0]
p = 11 (p mod 3 = 2):  HK_e(R,J) = [2, 363, 43923],  3p^(2e)-1 = [2, 362, 43922],  FS_e = [1, 0, 0],  Groebner FS_e = [1, 0, 0]


In [9]:
for p in [7, 5]:
    S.<x,y,z> = GF(p)[]
    print("p = %d:" % p, FS_function_diagonal(x^3 + y^3 + z^3))

p = 7: FS_e = 1   for e >= 0
p = 5: FS_e = 0   for e >= 1


The footnote of [Notes, Example 4.3] computes $\operatorname{HK}_1(R,J)=\alpha\big((2\delta_a+\delta_{a+1})^2(\delta_{2a}+2\delta_{2a+1})\big)=27a^2+18a+2=3p^2-1$ for $a=(p-1)/3$:

In [10]:
for p in [7, 13, 19, 31]:
    a = (p - 1) // 3
    d = lambda i: delta(i, p)
    print(p, ((2*d(a) + d(a+1))^2 * (d(2*a) + 2*d(2*a+1))).D(), 27*a^2 + 18*a + 2, 3*p^2 - 1)

7 146 146 146
13 506 506 506
19 1082 1082 1082
31 2882 2882 2882


### 1.6 F-signature: comparison with [CSTZ24]

The F-signature, compared with [CSTZ24, Theorem 7.1]:

In [11]:
p = 7
s = F_signature_diagonal(p, [3,3,3,3])
print("s(F_%s[x,y,z,t]/(x^3+y^3+z^3+t^3)) = %s" % (p, s))
'Does it coincide with the formula in Theorem 7.1 in [CSTZ]? ' + str(s == 3*p*(p-1)*(p+1)/(8*(3*p^3-p+2*(-1)^(p%3))))

s(F_7[x,y,z,t]/(x^3+y^3+z^3+t^3)) = 21/170


'Does it coincide with the formula in Theorem 7.1 in [CSTZ]? True'

The whole F-signature function (see §2 for how it is obtained):

In [12]:
S.<x,y,z,t> = GF(5)[]
FS_function_diagonal(x^3 + y^3 + z^3 + t^3)

FS_e = 15/124 * 5^(3e) + 109/124   for e >= 0

Check against quotient singularities, whose F-signature is $1/|G|$ when $p\nmid|G|$ [WY04, HL02]: $A_n=x^2+y^2+z^{n+1}$ ($|G|=n+1$), $E_6=x^2+y^3+z^4$ ($|G|=24$), $E_8=x^2+y^3+z^5$ ($|G|=120$).

In [13]:
p = 7
for name, exps, G in [("A_3", [2,2,4], 4), ("A_5", [2,2,6], 6), ("E_6", [2,3,4], 24), ("E_8", [2,3,5], 120)]:
    print(name, exps, " s =", F_signature_diagonal(p, exps), "  1/|G| =", 1/G)
S.<x,y,z> = GF(p)[]
print(FS_function_diagonal(x^2 + y^3 + z^5))

A_3 [2, 2, 4]  s = 1/4   1/|G| = 1/4
A_5 [2, 2, 6]  s = 1/6   1/|G| = 1/6
E_6 [2, 3, 4]  s = 1/24   1/|G| = 1/24
E_8 [2, 3, 5]  s = 1/120   1/|G| = 1/120
FS_e = 1/120 * 7^(2e) + 119/120   for even e >= 0
FS_e = 1/120 * 7^(2e) + 191/120   for odd e >= 1


## 2. Available functions

The polynomial is given directly, over a finite field `GF(p)`:

* `HK_function_diagonal(f)`: the Hilbert–Kunz function $e\mapsto\operatorname{HK}_e(R)$, one formula per residue class of $e$; the leading coefficient is $e_{\mathrm{HK}}(R)$. The result `H` can be evaluated, `H(e)`.
* `FS_function_diagonal(f)`: the F-signature function $e\mapsto\operatorname{FS}_e(R)$; the leading coefficient is the F-signature $s(R)$.
* `HK_groebner(f, e)`: $\operatorname{HK}_e(R)$ by a direct Gröbner-basis computation, for checks.

`HK_function_diagonal(f, ideal=[a_1,...,a_n])` replaces $\mathfrak m$ by $I=(x_1^{a_1},\dots,x_n^{a_n})$, so that $\beta=(a_1/d_1,\dots,a_n/d_n)$ in the notation of §3.

The lower-level functions take the exponent list `exps = [d_1, ..., d_n]` instead of the polynomial: `eHK_diagonal(p, exps, ideal=None, verbose=False)` (the multiplicity, with `verbose=True` printing the intermediate quantities $\mu$, $e_0$, $t$, $\ell^\sharp$), `F_signature_diagonal(p, exps)` (the F-signature), `find_period` and `_hk_value` (§3). The objects of the notes are also available: `delta_t(t, p)` is $\delta_t$, `D_k(p, ts)` is $D_k$ and `ell_k(p, rs)` is $\ell_k$ [Notes, Definition 3.1], and `phi_r(p, r, x)` is the polynomial $\phi_r$ of [Notes, Theorem 3.4].

**F-signature.** By Huneke–Leuschke [HL02] ([Notes, Proposition 4.1]), for a Gorenstein ring $R$ with a parameter ideal $I$ and a socle element $u$ of $R/I$:
$$\operatorname{FS}_e(R)=\ell(R/I)\,q^{\dim R}-\operatorname{HK}_e\big(R,I+(u)\big).$$
As in [Notes, Lemma 4.2], $I$ is generated by all variables but one, $x_j$; then $\ell(R/I)=d_j$, $u=x_j^{d_j-1}$, and $I+(u)=(x_1,\dots,x_j^{d_j-1},\dots,x_n)$. The notes take $j=n$; the code takes $x_j$ with the smallest exponent, so that $I$ is a minimal reduction of $\mathfrak m$ and $\ell(R/I)=e(R)=\min_id_i$ [Notes, after Lemma 4.2]. The first formula of [Notes, Proposition 4.1] holds for any parameter ideal, so both choices give the same function.

## 3. How it works

### 3.1 The Hilbert–Kunz function and the Hilbert–Kunz multiplicity

For a local ring $(R,\mathfrak m)$ of characteristic $p>0$ and dimension $d$, and an $\mathfrak m$-primary ideal $I$, the Hilbert–Kunz function is
$$\operatorname{HK}_e(R,I)=\ell\big(R/I^{[p^e]}\big),\qquad \operatorname{HK}_e(R)=\operatorname{HK}_e(R,\mathfrak m).$$
Monsky [Mon83] showed that $\operatorname{HK}_e(R,I)=c\,p^{de}+O(p^{(d-1)e})$ for a real constant $c=e_{\mathrm{HK}}(R,I)$, the **Hilbert–Kunz multiplicity**; $e_{\mathrm{HK}}(R)\ge1$ [Notes, §1].

### 3.2 The functions $D_k$ and $\ell_k$

Everything goes through the function [Notes, (1)]
$$D_k\Big(\frac{a_1}{d_1},\dots,\frac{a_n}{d_n}\Big)=\frac{1}{d_1\cdots d_n}\dim_k\frac{k[x_1,\dots,x_n]}{(x_1^{a_1},\dots,x_n^{a_n},x_1^{d_1}+\dots+x_n^{d_n})}.$$
In the Han–Monsky ring, $D_k(t_1,\dots,t_n)=\alpha(\delta_{t_1}\cdots\delta_{t_n})$, where $\delta_t$ interpolates linearly between $\delta_{\lfloor t\rfloor}$ and $\delta_{\lfloor t\rfloor+1}$ [Notes, Definition 3.1]; that the two descriptions agree is [HM93, Lemma 5.6] ([Notes, Lemma 3.2]). In particular, with $\beta=(a_1/d_1,\dots,a_n/d_n)$ and $I=(x_1^{a_1},\dots,x_n^{a_n})$,
$$\operatorname{HK}_e(R,I)=d_1\cdots d_n\,D_k(p^e\beta).$$

In [14]:
# [Notes, Lemma 3.2]: Groebner-basis dimension vs d_1...d_n D_k(a/d)
for p in [2, 3, 5]:
    for a, d in [((3,4), (2,3)), ((5,4,3), (3,2,4)), ((7,5,6), (3,3,2))]:
        S = PolynomialRing(GF(p), len(a), 'x'); X = S.gens()
        dim = S.ideal([X[i]^a[i] for i in range(len(a))] + [sum(X[i]^d[i] for i in range(len(a)))]).vector_space_dimension()
        print(p, a, d, dim, prod(d) * D_k(p, [a[i]/d[i] for i in range(len(a))]))

2 (3, 4) (2, 3) 7 7
2 (5, 4, 3) (3, 2, 4) 30 30
2 (7, 5, 6) (3, 3, 2) 66 66
3 (3, 4) (2, 3) 7 7
3 (5, 4, 3) (3, 2, 4) 30 30
3 (7, 5, 6) (3, 3, 2) 70 70
5 (3, 4) (2, 3) 7 7
5 (5, 4, 3) (3, 2, 4) 30 30
5 (7, 5, 6) (3, 3, 2) 66 66


The second function is $\ell_k(r_1,\dots,r_n)=\alpha(\lambda_{r_1}\cdots\lambda_{r_n})$ [Notes, Definition 3.1]. It is **$p$-multiplicative** [HM93, Theorem 4.5], [Notes, Theorem 3.3]: for $q=p^e$, $0\le k_i<q$ and $r_i\ge0$,
$$\ell_k(qr+k)=\begin{cases}\ell_k(r)\,\ell_k(k_1,\dots,k_n)&\sum r_i\text{ even},\\ \ell_k(r)\,\ell_k(q-1-k_1,k_2,\dots,k_n)&\sum r_i\text{ odd}.\end{cases}$$
A random test:

In [15]:
import random
random.seed(int(1))
ok = True
for p in [2, 3, 5]:
    for e in [1, 2]:
        q = p^e
        for _ in range(200):
            n = random.randint(2, 4)
            r = [random.randint(0, 5) for _ in range(n)]; k = [random.randint(0, q-1) for _ in range(n)]
            kk = k if sum(r) % 2 == 0 else [q-1-k[0]] + k[1:]
            ok = ok and ell_k(p, [q*r[i] + k[i] for i in range(n)]) == ell_k(p, r) * ell_k(p, kk)
ok

True

From this one gets the **functional equation** [HM93, Theorem 4.13], [Notes, Theorem 3.4]: for $q=p^e$, $u\in[0,q)^n$ and $r\in\mathbb Z_{\ge1}^n$,
$$D_k(qr+u)=\begin{cases}\ell_k(r)\,D_k(u_1,\dots,u_n)+q^{n-1}\phi_r(u/q)&\sum r_i\text{ even},\\ \ell_k(r)\,D_k(q-u_1,u_2,\dots,u_n)+q^{n-1}\phi_r(u/q)&\sum r_i\text{ odd},\end{cases}$$
with $\phi_r$ the polynomial of [Notes, Theorem 3.4] (`phi_r` in the code). A random test with rational $u$:

In [16]:
random.seed(int(2))
ok = True
for p in [2, 3, 5]:
    for e in [0, 1, 2]:
        q = p^e
        for _ in range(40):
            n = random.randint(2, 4)
            r = [random.randint(1, 4) for _ in range(n)]
            u = [min(QQ(random.randint(0, 6*q))/6, q - 1/6) for _ in range(n)]
            uu = u if sum(r) % 2 == 0 else [q - u[0]] + u[1:]
            lhs = D_k(p, [q*r[i] + u[i] for i in range(n)])
            rhs = ell_k(p, r) * D_k(p, uu) + q^(n-1) * phi_r(p, r, [x/q for x in u])
            ok = ok and lhs == rhs
ok

True

### 3.3 From the functional equation to the algorithm

The setup of [Notes, §3]: given $\beta\in\mathbb Q_{\ge0}^n$ and $\mu\ge1$, put
* $r=\lfloor\beta\rfloor$ and $z=\beta-r$;
* $R=\lfloor p^\mu\beta\rfloor$ and $z'=p^\mu\beta-R$;
* $t=R-p^\mu r$, so that $t=\lfloor p^\mu z\rfloor$;
* $\ell^\sharp=\ell_k(t)$ if $\sum r_i$ is even, and $\ell^\sharp=\ell_k(p^\mu-1-t_1,t_2,\dots,t_n)$ if $\sum r_i$ is odd, so that $\ell_k(R)=\ell^\sharp\,\ell_k(r)$ by [Notes, Theorem 3.3].

If $z'=z$ and $\sum R_i\equiv\sum r_i\pmod 2$, then $p^{-e(n-1)}\big(D_k(p^{e+\mu}\beta)-\ell^\sharp D_k(p^e\beta)\big)$ does not depend on $e$ [HM93, Theorem 5.2], [Notes, Theorem 3.5]. Consequently [Notes, Corollary 3.6]
$$\operatorname{HK}_{e+\mu}(R,I)-\ell^\sharp\,\operatorname{HK}_e(R,I)=A\,p^{e(n-1)}\quad\text{and}\quad \operatorname{HK}_e(R,I)=C\,p^{(n-1)e}+\Delta_e\ \text{ with }\ \Delta_{e+\mu}=\ell^\sharp\Delta_e,$$
for $e\ge e_0$ and constants $A$, $C$.

**Notation in the code.**

| Symbol in code | Meaning |
|---|---|
| `p` | the prime characteristic |
| `exps` | the exponent vector $(d_1,\dots,d_n)$ of the diagonal hypersurface |
| `ideal` | exponents $(a_1,\dots,a_n)$ of the $\mathfrak m$-primary ideal $I=(x_1^{a_1},\dots,x_n^{a_n})$ (default all $1$, i.e. $I=\mathfrak m$) |
| `dim`  | the Krull dimension $\dim R=n-1$ |
| `new_beta` | $\beta=p^{e_0}(a_1/d_1,\dots,a_n/d_n)$ (§3.4) |
| `r`, `z`, `t` | $r$, $z$, $t$ as above |
| `lsh` | $\ell^\sharp$ |
| `mu` | $\mu$ |
| `e0` | $e_0$, the threshold from which the closed form is valid |
| `lambda_matrix(p,e,k)` | the multiplication matrix of $\lambda_k\in\Lambda_e$ (notebook 01) |

The algorithm has three layers:
1. **The Han–Monsky ring** (notebook 01): $\ell^\sharp$ is the $(0,0)$ entry of a product of $\lambda$-matrices;
2. **Period search** (§3.4): find $e_0$ and $\mu$;
3. **Assembly** (§3.5): solve a small linear system for $C=e_{\mathrm{HK}}(R,I)$ and $\Delta_e$.

> **References.** The technique was introduced in [HM93], building on Han's thesis [Han91]. The theory of the Han–Monsky ring was later developed by Monsky and Teixeira [MT04, MT06]. The matrix representation is from [Yos09].

### 3.4 Finding the period $\mu$ and the threshold $e_0$

This is the number-theoretic input of [HM93, §5] ([Notes, §3.1, Lemma 3.7]), implemented in `find_period`. Let $\tilde\beta=(a_1/d_1,\dots,a_n/d_n)$.

1. **Strip $p$ from the denominators.** $e_0$ is the largest exponent with $p^{e_0}\mid d_i$ for some $i$, and $\beta=p^{e_0}\tilde\beta$, so that no denominator of $\beta$ is divisible by $p$. Then $\operatorname{HK}_{e_0+e}(R,I)=d_1\cdots d_n\,D_k(p^e\beta)$.
2. **Search for the smallest $\mu\ge1$** for which the hypotheses of [Notes, Theorem 3.5] hold, i.e. $z'=z$ and $\sum R_i\equiv\sum r_i\pmod 2$. The code accepts a slightly larger set of $\mu$'s, following its source [HM93, §5]: $z'$ may also be obtained from $z$ by changes of sign mod 1 in an even number of coordinates (with the same parity condition) or in an odd number of coordinates (with the opposite parity). For $\mu$ a common multiple of the multiplicative orders of $p$ modulo the denominators of $\beta$ one has $z'=z$ [Notes, Lemma 3.7(a)]; the parity condition is discussed in [Notes, Lemma 3.7(b),(c)]. The search stops at the a priori bound $\operatorname{lcm}\varphi(\text{denominators})$ (`max_mu` in the code).

The function returns $(\mu,e_0,\beta)$; `verbose=True` prints the search.

In [17]:
find_period(5, [3,3,3,3], [1,1,1,2], talk=True)

'Period mu = 1, e0 = 0'

In [18]:
find_period(3, [6,6,6,5], verbose=True)

Maximum mu: 4
mu=1  z'=[1, 1, 1, 4]  z~z'=False  z*~z'=False  parity=True
mu=2  z'=[1, 1, 1, 2]  z~z'=True  z*~z'=True  parity=False


(2, 1, (1/2, 1/2, 1/2, 3/5))

### 3.5 Assembling the Hilbert–Kunz function

With the period in hand, `eHK_diagonal` computes:

- $\ell^\sharp$ (`lsh`): $t=\lfloor p^\mu z\rfloor$, with $t_1$ replaced by $p^\mu-1-t_1$ if $\sum r_i$ is odd, and $\ell^\sharp=\ell_k(t)=\alpha(\lambda_{t_1}\cdots\lambda_{t_n})$, which is the $(0,0)$ entry of the product of the matrices of the $\lambda_{t_i}$ in $\Lambda_\mu$;
- two values of the Hilbert–Kunz function, $\operatorname{HK}_{e_0}(R,I)$ and $\operatorname{HK}_{e_0+\mu}(R,I)$, computed **by brute force** (`_hk_value`) as $\mathbb F_p$-dimensions of the quotient. They could also be obtained from the matrices, but the matrices grow fast and there is no practical gain.

By [Notes, Corollary 3.6], writing $\Delta_e=-y$ these satisfy the $2\times2$ linear system
$$\operatorname{HK}_{e_0}(R,I)=C\,p^{(n-1)e_0}-y,\qquad \operatorname{HK}_{e_0+\mu}(R,I)=C\,p^{(n-1)(e_0+\mu)}-y\,\ell^\sharp,$$
whose solution gives the multiplicity $C=e_{\mathrm{HK}}(R,I)$ and the correction term; the remaining residue classes of $e$ mod $\mu$ are filled in with $\operatorname{HK}_{e_0+1},\dots,\operatorname{HK}_{e_0+\mu-1}$.

In [19]:
p, e = 7, 1
float(_hk_value(p, [3,3,3,3], [1,1,1,2], e) / p^(3*e))

2.868804664723032

In [20]:
eHK_diagonal(5, [4,4,4,4], verbose=True)

Original beta: (1/4, 1/4, 1/4, 1/4)
mu = 1 , e0 = 0 , new beta = (1/4, 1/4, 1/4, 1/4)
sum(r) odd: False (sum r = 0 )
t-vector = (1, 1, 1, 1)  so l# = 3


168/61

**Checking [Notes, Corollary 3.6].** The quantity $\big(\operatorname{HK}_{e+\mu}(R)-\ell^\sharp\operatorname{HK}_e(R)\big)/p^{e(n-1)}$ is constant for $e\ge e_0$; it equals $C(p^{\mu(n-1)}-\ell^\sharp)$. For instance, for $x^4+y^4+z^4+t^4$ over $\mathbb F_5$ it is $\frac{168}{61}(125-3)=336$.

In [21]:
for p, exps in [(5,[4,4,4,4]), (3,[6,6,6,5]), (7,[3,3,3,3])]:
    D = eHK_diagonal(p, exps, _data=True)
    mu, e0, lsh, n = D['mu'], D['e0'], D['lsh'], len(exps)
    HK = [_hk_value(p, exps, None, e) for e in range(e0 + mu + 3)]
    print(p, exps, " mu =", mu, " e0 =", e0, " l# =", lsh, " A =",
          [(HK[e+mu] - lsh*HK[e]) / p^(e*(n-1)) for e in range(e0, e0 + 3)],
          " C(p^(mu(n-1)) - l#) =", D['eHK'] * (p^(mu*(n-1)) - lsh))

5 [4, 4, 4, 4]  mu = 1  e0 = 0  l# = 3  A = [336, 336, 336]  C(p^(mu(n-1)) - l#) = 336
3 [6, 6, 6, 5]  mu = 2  e0 = 1  l# = 3  A = [2754, 2754, 2754]  C(p^(mu(n-1)) - l#) = 2754
7 [3, 3, 3, 3]  mu = 1  e0 = 0  l# = 5  A = [678, 678, 678]  C(p^(mu(n-1)) - l#) = 678


## References

Labels follow the alpha style of the notes where possible; the BibTeX key used in `han_monsky.tex` is given in brackets, so each entry can be matched with the notes' `.bib` file.

- **[Notes]** J. Castillo-Rey. *The Hilbert–Kunz function and F-signature function of the diagonal hypersurfaces*, notes (`han_monsky.tex`). Numbering of results as in the compiled notes.
- **[CSTZ24]** A. Caminata, S. Shideler, K. Tucker, F. Zerman. *F-signature functions of diagonal hypersurfaces*, arXiv:2403.12863 (2024).  [`shidelerFsignatureFunctionsDiagonal2024`]
- **[Han91]** C. Han. *The Hilbert–Kunz function of a diagonal hypersurface*, Ph.D. thesis, Brandeis University, 1991.  [`han1992hilbert`]
- **[HL02]** C. Huneke and G. J. Leuschke. *Two theorems about maximal Cohen–Macaulay modules*, Math. Ann. **324** (2002), no. 2, 391–404.  [`hunekeTwoTheoremsMaximal2004`]
- **[HM93]** C. Han and P. Monsky. *Some surprising Hilbert–Kunz functions*, Math. Z. **214** (1993), no. 1, 119–135. DOI: [10.1007/BF02572395](https://doi.org/10.1007/BF02572395).  [`hanmonskySurprising1993`]
- **[Mon83]** P. Monsky. *The Hilbert–Kunz function*, Math. Ann. **263** (1983), 43–49.  [`monskyHilbertKunzFunction1983`]
- **[MT04]** P. Monsky and P. Teixeira. *p-Fractals and power series — I. Some 2-variable results*, J. Algebra **280** (2004), 505–536. DOI: [10.1016/j.jalgebra.2004.05.016](https://doi.org/10.1016/j.jalgebra.2004.05.016).  [`monsky2004p`]
- **[MT06]** P. Monsky and P. Teixeira. *p-Fractals and power series — II. Some applications to Hilbert–Kunz theory*, J. Algebra **304** (2006), 237–255 (arXiv:math/0506584).  [`monskyPFractals2006`]
- **[WY04]** K.-i. Watanabe and K.-i. Yoshida. *Minimal relative Hilbert–Kunz multiplicity*, Illinois J. Math. **48** (2004), 273–294.
- **[Yos09]** K.-i. Yoshida. *Small Hilbert–Kunz multiplicity and $(A_1)$-type singularity*, Proceedings of the 4th Japan-Vietnam Joint Seminar on Commutative Algebra, Meiji University (2009).  [`Yo09`]